<a href="https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule idea

I will prioritize content pages that show both a meaningful search opportunity and signs of content staleness.

The rule will use two observed signals from the March 2026 development window:

1. **Staleness:** how many days have passed since the content was last updated, measured at the end of March 2026.
2. **Search impressions:** the total observed search impressions for the page during March 2026.

The baseline score will give higher priority to pages with higher search impressions and greater staleness. The output is a decision-support ranking for human review, not an automatic decision that a page must be refreshed.

### Reason code

The rule will use one reason code:

* `STALE_HIGH_OPPORTUNITY` — the page is relatively stale and has meaningful observed search exposure.

### Action label

The action label for the baseline queue is:

* `REVIEW_REFRESH` — review the page for a possible content refresh.

The thresholds will be chosen from the observed March data after checking the two signals, rather than assuming that the signals are useful in advance.


In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

repo_id = "FlyRank/internship-warehouse"

march_path = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

content_path = hf_hub_download(
    repo_id=repo_id,
    filename="dim_content.parquet",
    repo_type="dataset"
)

march_df = pd.read_parquet(march_path)
dim_content = pd.read_parquet(content_path)

march_features = (
    march_df.groupby("content_hash_id")
    .agg(
        march_impressions=("gsc_impressions", "sum")
    )
    .reset_index()
)

dim_content["content_updated_date"] = pd.to_datetime(
    dim_content["content_updated_date"],
    errors="coerce"
)

updated_dates = (
    dim_content[["content_hash_id", "content_updated_date"]]
    .drop_duplicates("content_hash_id")
)

march_features = march_features.merge(
    updated_dates,
    on="content_hash_id",
    how="left"
)

decision_date = pd.Timestamp("2026-03-31")

march_features["staleness_days"] = (
    decision_date - march_features["content_updated_date"]
).dt.days

signal_df = march_features[
    ["content_hash_id", "march_impressions", "staleness_days"]
].copy()

print("Signal dataset shape:", signal_df.shape)
display(signal_df.head())

Signal dataset shape: (331437, 3)


,content_hash_id,march_impressions,staleness_days
0,content_000005d4ced12088,86,-48
1,content_00001e488b74b799,0,-50
2,content_00007bd2985b77c3,47,34
3,content_00008950670cb6b5,0,-50
4,content_0000a348850eb1fc,0,-50


In [ ]:
future_updates = signal_df["staleness_days"] < 0

print("Pages with future update dates:", future_updates.sum())
print("Pages with valid staleness:", (~future_updates).sum())

Pages with future update dates: 293358
Pages with valid staleness: 38079


In [ ]:
signal_df["staleness_days"] = signal_df["staleness_days"].where(
    signal_df["staleness_days"] >= 0
)

print("Valid staleness values:", signal_df["staleness_days"].notna().sum())
print("Missing staleness values:", signal_df["staleness_days"].isna().sum())
print("Minimum valid staleness:", signal_df["staleness_days"].min())
print("Maximum valid staleness:", signal_df["staleness_days"].max())

Valid staleness values: 38079
Missing staleness values: 293358
Minimum valid staleness: 7.0
Maximum valid staleness: 303.0


In [ ]:
staleness_check = signal_df[
    signal_df["staleness_days"].notna()
].copy()

staleness_check["staleness_bucket"] = pd.cut(
    staleness_check["staleness_days"],
    bins=[0, 30, 60, 120, float("inf")],
    labels=["7-30 days", "31-60 days", "61-120 days", "121+ days"]
)

staleness_bucket_table = (
    staleness_check
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        avg_impressions=("march_impressions", "mean"),
        median_impressions=("march_impressions", "median")
    )
    .reset_index()
)

display(staleness_bucket_table)

,staleness_bucket,n,avg_impressions,median_impressions
0,7-30 days,975,100.870769,3.0
1,31-60 days,28598,1184.215540,245.0
2,61-120 days,1307,43.949503,0.0
3,121+ days,7199,66.641200,0.0


### Signal 1 — Staleness

**Verdict: MIXED**

The March 2026 bucket table shows that staleness is not a simple monotonic signal of search opportunity. Pages updated 31–60 days ago have the highest average and median impressions, while pages in the older 61–120 and 121+ day buckets have much lower observed impressions.

This means staleness may help identify pages for review, but older content is not automatically a higher-opportunity page. I will therefore use staleness as a supporting signal rather than treating it as sufficient evidence for a refresh.


In [ ]:
impressions_check = signal_df.copy()

impressions_check["impressions_bucket"] = "0 impressions"

positive_impressions = impressions_check["march_impressions"] > 0

impressions_check.loc[positive_impressions, "impressions_bucket"] = pd.qcut(
    impressions_check.loc[positive_impressions, "march_impressions"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

impressions_bucket_table = (
    impressions_check
    .groupby("impressions_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        avg_impressions=("march_impressions", "mean"),
        median_impressions=("march_impressions", "median")
    )
    .reset_index()
)

display(impressions_bucket_table)

,impressions_bucket,n,avg_impressions,median_impressions
0,0 impressions,154699,0.000000,0.0
1,High,58887,4539.272573,2041.0
2,Low,59013,12.283734,7.0
3,Medium,58838,214.632465,174.0


### Signal 2 — Search Impressions

**Verdict: CONFIRMED**

The March 2026 bucket table shows a clear separation in observed search exposure. Pages in the High impressions bucket have much higher average and median impressions than the Medium and Low buckets, while 154,699 pages have zero impressions.

This supports using March impressions as a search-opportunity signal. Higher observed impressions indicate greater observed search exposure, although impressions alone do not prove that a page needs a refresh or that refreshing it will improve performance.


### Baseline scoring rule

I will rank content pages using two observed signals from March 2026:

* Higher March impressions indicate higher observed search opportunity.
* Higher valid staleness indicates older content.

The baseline score gives 70% weight to search opportunity and 30% weight to staleness.

Pages with a score of 70 or higher receive the action `REVIEW_REFRESH`. Pages below the threshold receive the action `MONITOR`. The baseline rule uses the single reason code `STALE_HIGH_OPPORTUNITY`.

The score is used for human review prioritization and does not automatically determine that a page should be refreshed.


In [ ]:
import numpy as np
import os

queue_df = signal_df.copy()

queue_df["opportunity_score"] = (
    queue_df["march_impressions"].rank(method="average", pct=True) * 100
)

queue_df["staleness_score"] = np.nan

valid_stale = queue_df["staleness_days"].notna()

queue_df.loc[valid_stale, "staleness_score"] = (
    queue_df.loc[valid_stale, "staleness_days"]
    .rank(method="average", pct=True) * 100
)

queue_df["staleness_score"] = queue_df["staleness_score"].fillna(0)

queue_df["baseline_score"] = (
    0.70 * queue_df["opportunity_score"]
    + 0.30 * queue_df["staleness_score"]
)

queue_df["action"] = np.where(
    queue_df["baseline_score"] >= 70,
    "REVIEW_REFRESH",
    "MONITOR"
)

queue_df["reason_code"] = "STALE_HIGH_OPPORTUNITY"

queue_df = queue_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue_df["rank"] = queue_df.index + 1

baseline_queue = queue_df[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "march_impressions",
        "staleness_days"
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("Queue rows:", len(baseline_queue))
print("Output:", output_path)

display(baseline_queue.head(10))

Queue rows: 331437
Output: work/outputs/baseline_action_score.csv


,rank,content_hash_id,baseline_score,action,reason_code,march_impressions,staleness_days
0,1,content_42ce26be1ec6be00,95.704131,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,4411,264.0
1,2,content_bea86ce3455100b0,94.513173,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,3670,232.0
2,3,content_097459d155cccb26,94.225199,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,37930,124.0
3,4,content_f2df5a8a9057783e,94.209147,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,35980,124.0
4,5,content_ac4e2d9d3bbb06de,94.184859,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,33348,124.0
5,6,content_66d1fffc91f4f029,94.162683,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,31038,124.0
6,7,content_9598a57544925111,94.018801,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,24066,123.0
7,8,content_b956947c822af734,94.001325,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,22502,124.0
8,9,content_19daa2f24df1882d,93.986588,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,4968,176.0
9,10,content_0d2aaf57d7146812,93.935587,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,21060,123.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["staleness_days"].notna(),
    "Moderate confidence: both observed impressions and valid staleness are available.",
    "Lower confidence: staleness is unavailable at the decision moment."
)

top20["what_would_make_it_wrong"] = np.where(
    top20["march_impressions"] == 0,
    "The page has no observed search exposure, so refresh priority may be weak.",
    "The page may have high exposure but no actual content problem; impressions alone do not prove that a refresh is needed."
)

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "baseline_score",
            "march_impressions",
            "staleness_days",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,action,reason_code,baseline_score,march_impressions,staleness_days,confidence_note,what_would_make_it_wrong
0,1,content_42ce26be1ec6be00,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,95.704131,4411,264.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
1,2,content_bea86ce3455100b0,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,94.513173,3670,232.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
2,3,content_097459d155cccb26,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,94.225199,37930,124.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
3,4,content_f2df5a8a9057783e,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,94.209147,35980,124.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
4,5,content_ac4e2d9d3bbb06de,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,94.184859,33348,124.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
5,6,content_66d1fffc91f4f029,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,94.162683,31038,124.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
6,7,content_9598a57544925111,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,94.018801,24066,123.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
7,8,content_b956947c822af734,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,94.001325,22502,124.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
8,9,content_19daa2f24df1882d,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,93.986588,4968,176.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...
9,10,content_0d2aaf57d7146812,REVIEW_REFRESH,STALE_HIGH_OPPORTUNITY,93.935587,21060,123.0,Moderate confidence: both observed impressions...,The page may have high exposure but no actual ...


### Weak picks

The weakest picks are pages with zero March search impressions and no valid staleness signal. These pages receive a low baseline score and are assigned the `MONITOR` action. They are weak refresh candidates because there is little observed search exposure to justify prioritizing a refresh.

### Leakage check

The baseline rule uses only information available within the March 2026 development window: March search impressions and content staleness measured at the end of March.

No future-window data, declining labels, April–June performance, or product flags are used in the scoring rule.


In [ ]:
# Check the weakest ranked pages and verify that no future-window
# or label-derived inputs are used in the baseline rule.

weak_picks = baseline_queue.tail(10).copy()

print("Weakest 10 ranked pages:")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_score",
            "action",
            "reason_code",
            "march_impressions",
            "staleness_days"
        ]
    ]
)

print("\nLeakage check:")

used_columns = {
    "march_impressions",
    "staleness_days"
}

future_or_label_terms = [
    "label",
    "declining",
    "future",
    "april",
    "may",
    "june",
    "click"
]

print("Signals used:", sorted(used_columns))

print(
    "Future/label-derived fields used:",
    [
        col for col in used_columns
        if any(term in col.lower() for term in future_or_label_terms)
    ]
)

print("Leakage check passed: March impressions and observed staleness only.")

Weakest 10 ranked pages:


,rank,content_hash_id,baseline_score,action,reason_code,march_impressions,staleness_days
331427,331428,content_74f498362c8d7217,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331428,331429,content_31d6b6169943b29b,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331429,331430,content_31d732e5b7d86f68,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331430,331431,content_31d73546e6a02a5c,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331431,331432,content_31d75f2f21564871,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331432,331433,content_31d7630f1644f770,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331433,331434,content_74f4168505ee9c6c,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331434,331435,content_74f42d2c8c03bed6,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331435,331436,content_74f45ff43b96f003,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331436,331437,content_31d6725775baaa57,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN



Leakage check:
Signals used: ['march_impressions', 'staleness_days']
Future/label-derived fields used: []
Leakage check passed: March impressions and observed staleness only.


In [ ]:
# Check the weakest ranked pages and verify that no future-window
# or label-derived inputs are used in the baseline rule.

weak_picks = baseline_queue.tail(10).copy()

print("Weakest 10 ranked pages:")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_score",
            "action",
            "reason_code",
            "march_impressions",
            "staleness_days"
        ]
    ]
)

print("\nLeakage check:")

used_columns = {
    "march_impressions",
    "staleness_days"
}

future_or_label_terms = [
    "label",
    "declining",
    "future",
    "april",
    "may",
    "june",
    "click"
]

print("Signals used:", sorted(used_columns))

print(
    "Future/label-derived fields used:",
    [
        col for col in used_columns
        if any(term in col.lower() for term in future_or_label_terms)
    ]
)

print("Leakage check passed: March impressions and observed staleness only.")

Weakest 10 ranked pages:


,rank,content_hash_id,baseline_score,action,reason_code,march_impressions,staleness_days
331427,331428,content_74f498362c8d7217,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331428,331429,content_31d6b6169943b29b,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331429,331430,content_31d732e5b7d86f68,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331430,331431,content_31d73546e6a02a5c,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331431,331432,content_31d75f2f21564871,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331432,331433,content_31d7630f1644f770,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331433,331434,content_74f4168505ee9c6c,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331434,331435,content_74f42d2c8c03bed6,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331435,331436,content_74f45ff43b96f003,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN
331436,331437,content_31d6725775baaa57,16.336438,MONITOR,STALE_HIGH_OPPORTUNITY,0,NaN



Leakage check:
Signals used: ['march_impressions', 'staleness_days']
Future/label-derived fields used: []
Leakage check passed: March impressions and observed staleness only.


## Self-check

Before you submit, confirm each line honestly:

* Every section above is filled with both markdown reasoning and supporting code.
* The notebook runs top to bottom without errors.
* No client names, URLs, or private queries are included.
* Claims use careful language such as observed, measured, directional, and decision-support.
* The notebook is committed under `work/notebooks/`.
